# 📋 Trading Research Checklist
**Experiment Name:**  
**Version / Tag:**  
**Date:**  
**Dataset Snapshot:**  
**Notes:**  

---

## PHASE 0 — DATA & EVENT INTEGRITY (HARD GATE)

- [ ] `event_ts ∈ df.index` rate ≥ 99.9%
- [ ] No forward-looking fields used
- [ ] ATR, compression_score, volume shock computed from same bars
- [ ] No unintended mixed timeframes
- [ ] Event overlap handled or prevented
- [ ] Minimum spacing rule documented
- [ ] Entry price definition clear (e.g. next bar open)
- [ ] Sanity checks:
  - [ ] `mfe < 0 == 0`
  - [ ] `mae > 0` allowed

**Pass / Fail:**  
**Comments:**

---

## PHASE 1 — BASELINE DEFINITION (ANCHOR)

- [ ] Baseline exit logic defined and simple
- [ ] Baseline parameters frozen
- [ ] Baseline metrics recorded:
  - [ ] trades
  - [ ] win rate
  - [ ] mean / median return
  - [ ] total return
  - [ ] max drawdown
- [ ] Baseline never modified after this phase

**Pass / Fail:**  
**Comments:**

---

## PHASE 2 — MECHANISM ISOLATION

- [ ] Only ONE new idea introduced
- [ ] No entry changes
- [ ] No sizing changes
- [ ] Same-trades comparison enforced
- [ ] Merge keys verified: `(symbol, entry_ts, event_ts)`
- [ ] Delta stats computed (`new - baseline`)
- [ ] Median delta ≥ 0

**Pass / Fail:**  
**Comments:**

---

## PHASE 3 — STRUCTURAL COHERENCE

- [ ] Exit reasons interpretable
- [ ] Winners dominated by logical exits (trail / time)
- [ ] Losers dominated by stops
- [ ] Compression buckets behave as expected:
  - [ ] Q1/Q2 contribute majority of profit
  - [ ] Q4 not catastrophic
- [ ] All parameters explainable (ATR / bars)

**Pass / Fail:**  
**Comments:**

---

## PHASE 4 — ROBUSTNESS TESTS (ANTI-OVERFIT)

### Time Randomization
- [ ] ±1 bar entry shift
- [ ] ≥20 randomized runs
- [ ] Median performance > baseline

### Symbol Holdout
- [ ] ≥30% symbols excluded from tuning
- [ ] Same parameters used
- [ ] Directional improvement holds

### Bucket Stability
- [ ] Q1 > Q2 > Q3 ordering mostly preserved
- [ ] No wild sign flips

**Pass / Fail:**  
**Comments:**

---

## PHASE 5 — REGIME & TAIL RISK

- [ ] Worst 5 trades reviewed
- [ ] Worst symbol reviewed
- [ ] Worst bucket reviewed
- [ ] Single-trade loss bounded
- [ ] Dead-trade logic present
- [ ] No runaway losers

**Pass / Fail:**  
**Comments:**

---

## PHASE 6 — COMPLEXITY BUDGET

- [ ] Exit logic line-count within limit
- [ ] ≤5 free parameters per mechanism
- [ ] Parameters mostly derived, not tuned
- [ ] New rules justified by removing old ones

**Pass / Fail:**  
**Comments:**

---

## PHASE 7 — PRE-PRODUCTION FREEZE

- [ ] Version tag created
- [ ] Code + params checksummed
- [ ] Dataset snapshot stored
- [ ] System description written (1 page)
- [ ] Assumptions listed
- [ ] Known failure modes documented
- [ ] Kill criteria defined

**Pass / Fail:**  
**Comments:**

---

## FINAL DECISION

- [ ] APPROVE FOR NEXT STAGE
- [ ] HOLD (needs fixes)
- [ ] REJECT (overfit / fragile)

**Summary Rationale:**

---

## POST-APPROVAL NOTES (Optional)

- Deployment considerations:
- Monitoring metrics:
- Future research ideas:



# Exit Policy v1

**System:** Long-Only Volume Shock Strategy
**Scope:** Trade exit behavior after a validated long entry
**Status:** Locked (no tuning during evaluation)

---

## 1. Objective

The exit system’s purpose is to:

* Protect capital from invalid or non-performing trades
* Allow structurally strong trades to compound
* Recycle capital quickly from weak regimes
* Behave deterministically across all market states

The exit logic is **compression-aware**, but **entry-agnostic**.

---

## 2. Core Principles

1. **Risk First**
   Catastrophic loss prevention overrides all other objectives.

2. **Structure Over Profit Targets**
   Trades are exited based on loss of structure, not arbitrary price levels.

3. **Compression Determines Trust**
   Deeper compression implies higher continuation probability and longer holding permission.

4. **Time Is a Cost**
   Trades that fail to progress within a defined window are exited regardless of P&L.

5. **Simplicity Over Optimization**
   No discretionary exits, partial exits, or profit targets.

---

## 3. Trade Lifecycle Overview

All trades follow this sequence:

1. Entry at `event_ts + 1 bar`
2. Phase-A structural validation
3. Risk containment
4. Performance assessment
5. Trend participation (if earned)
6. Time-based termination

At no point may a trade bypass an earlier safety condition.

---

## 4. Exit Components (Ordered by Priority)

### 4.1 Hard Stop (Non-Negotiable)

* Fixed ATR-based stop from entry
* Active immediately
* Applies to all trades and all regimes

**Purpose:** Catastrophic risk control

---

### 4.2 Phase-A Validation Gate

* Evaluated in early bars post-entry
* Requires minimum favorable excursion relative to ATR
* Optional early stop if structure invalidates immediately

**Purpose:** Reject false breakouts and event noise

---

### 4.3 Dead Trade Rule

* If trade fails to reach minimum MFE within `DEAD_BARS`
* Exit regardless of unrealized P&L

**Purpose:** Capital efficiency and variance reduction

---

### 4.4 Trailing Exit (Conditional)

Trailing logic activates **only after**:

* Favorable excursion exceeds activation threshold
* Trade demonstrates continuation behavior

Trailing behavior:

* Anchored to MFE
* Exits on defined giveback
* Parameters derived from compression severity

**Purpose:** Capture asymmetric upside without prediction

---

### 4.5 Time Cap (Final Authority)

* Absolute maximum holding period
* Bucket-dependent:

  * Deeper compression → longer permission
  * Shallow compression → faster release

**Purpose:** Prevent capital stagnation and regime drift

---

## 5. Compression-Aware Policy

Compression score at entry determines **holding permission**, not entry validity.

| Compression Bucket | Policy                              |
| ------------------ | ----------------------------------- |
| Q1 (Deep)          | Full trailing, longest time cap     |
| Q2                 | Full trailing, moderate time cap    |
| Q3                 | Limited trailing or early time exit |
| Q4 (Shallow)       | Fast exit only (time or baseline)   |

No trade may upgrade its permission level after entry.

---

## 6. Explicit Non-Goals

The exit system does **not**:

* Predict tops
* Optimize profit per trade
* Use fixed profit targets
* Scale out positions
* React to indicators not present at entry

---

## 7. Failure Handling & Kill Criteria

The system must be reviewed or disabled if:

* Median return turns negative for a full evaluation window
* Max drawdown exceeds predefined risk tolerance
* Q4 bucket contributes disproportionate losses
* Exit reasons shift toward hard stops in high compression regimes

---

## 8. Change Control

* Any change to exit behavior requires:

  * New version tag
  * Full baseline comparison
  * Checklist re-validation
* No mid-evaluation parameter changes permitted

---

## 9. Summary Statement

> This exit policy prioritizes **structural validity**, **capital protection**, and **regime awareness** over profit maximization.
> Profits are treated as a *consequence* of correct exits, not the objective.

---

### Version History

* **v1.0** — Initial locked policy (compression-aware trailing)